# 03 — Fine-tuning XLM-RoBERTa
Multi-task: sentimento, categoria, rating, prioridade


In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data.loader import load_processed, split_dataset
from src.models.trainer import train
from src.models.classifier import ReviewInference, CATEGORIES
from src.utils.metrics import print_report, get_confusion_matrix

print(f'PyTorch: {torch.__version__}')
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Carregar dados processados

In [ ]:
df = load_processed('reviews_labeled.csv')
print(f'Total: {len(df):,} reviews')

# Garantir colunas de categoria existem
cat_cols = [f'cat_{c}' for c in CATEGORIES]
missing_cats = [c for c in cat_cols if c not in df.columns]
if missing_cats:
    print(f'Colunas ausentes, executando add_category_labels...')
    from src.data.preprocessor import add_category_labels
    df = add_category_labels(df)

train_df, val_df, test_df = split_dataset(df)
print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

## 2. Treinar o modelo

In [ ]:
# Para testar rapidamente use um subset menor
# Remova o sample() para treinar no dataset completo
QUICK_TEST = False
if QUICK_TEST:
    train_df = train_df.sample(2000, random_state=42)
    val_df = val_df.sample(500, random_state=42)

model = train(
    train_df=train_df,
    val_df=val_df,
    model_name='xlm-roberta-base',
    epochs=3,
    batch_size=16,      # Reduza para 8 se tiver OOM
    lr=2e-5,
    max_length=256,
)

## 3. Avaliar no conjunto de teste

In [ ]:
from src.models.trainer import evaluate, MultiTaskLoss, ReviewDataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')
test_ds = ReviewDataset(test_df, tokenizer)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)
criterion = MultiTaskLoss()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
test_loss, test_metrics = evaluate(model, test_loader, criterion, device)

print(f'Test Loss: {test_loss:.4f}')
for k, v in test_metrics.items():
    print(f'  {k}: {v:.4f}')

## 4. Relatório detalhado de sentimento

In [ ]:
import torch

model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(batch['input_ids'], batch['attention_mask'])
        all_preds.extend(out['sentiment'].argmax(dim=-1).cpu().tolist())
        all_true.extend(batch['sentiment'].cpu().tolist())

print_report(all_true, all_preds, target_names=['negativo', 'neutro', 'positivo'])

cm = get_confusion_matrix(all_true, all_preds)
import seaborn as sns
plt.figure(figsize=(7,5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=['neg','neu','pos'], yticklabels=['neg','neu','pos'])
plt.title('Confusion Matrix — Sentimento')
plt.ylabel('Real')
plt.xlabel('Predito')
plt.show()

## 5. Teste de inferência manual

In [ ]:
engine = ReviewInference(checkpoint='../models/saved/best_model.pt')

test_reviews = [
    "The room was dirty and the staff was very rude. Terrible experience.",
    "Excelente hotel! Quarto limpo, café da manhã delicioso e localização perfeita.",
    "Hotel was okay, nothing special. The wifi was slow but the bed was comfortable.",
]

results = engine.predict_batch(test_reviews)
for r in results:
    print(f"Review: {r['text'][:80]}...")
    print(f"  Sentimento: {r['sentiment']} | Rating: {r['rating_predicted']} | Prioridade: {r['priority']}")
    print(f"  Categorias: {r['categories']}\n")